# ARC Visualization

这个 notebook 提供三类 ARC 标准可视化：

1. 按题号可视化 `/home/lixinyu/snap/NVARC/ARC-data/ARC-AGI-2` 中的 ARC 题目。
2. 对 `/home/lixinyu/snap/new_code/solvers/arc2` 按题号可视化对应 solver，而不是按整个路径批量可视化。
3. 对 `new/variants` 使用用户提供的目录路径，递归查找该路径下所有 `code.py` 并进行可视化。

所有绘图都使用 ARC 0-9 标准颜色。

In [1]:
from __future__ import annotations

import hashlib
import importlib.util
import inspect
import json
import random
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from matplotlib.colors import BoundaryNorm, ListedColormap

BASE_DIR = Path("/home/lixinyu/snap")
ARC_DATA_ROOT = BASE_DIR / "NVARC/ARC-data/ARC-AGI-2"
ARC1_DATA_ROOT = BASE_DIR / "NVARC/ARC-data/ARC-AGI/data"
SOLVER_ROOT = BASE_DIR / "/home/lixinyu/snap/new/arc2_solver_select"
ARC2_ROOT = BASE_DIR / "/home/lixinyu/snap/new/arc2_solver_select"
VARIANTS_ROOT = BASE_DIR / "new/variants"
ARC2_SOLVER_SELECT = BASE_DIR / "new/arc2_solver_select"

for extra_path in (SOLVER_ROOT, ARC2_ROOT, ARC2_SOLVER_SELECT):
    extra_path_str = str(extra_path.resolve())
    if extra_path_str not in sys.path:
        sys.path.insert(0, extra_path_str)

ARC_COLORS = [
    "#000000",
    "#0074D9",
    "#FF4136",
    "#2ECC40",
    "#FFDC00",
    "#AAAAAA",
    "#F012BE",
    "#FF851B",
    "#7FDBFF",
    "#870C25",
]
ARC_CMAP = ListedColormap(ARC_COLORS)
ARC_NORM = BoundaryNorm(np.arange(-0.5, 10.5, 1), ARC_CMAP.N)

In [2]:
CELL_SIZE_INCH = 0.1
LEFT_MARGIN_INCH = 0.45
RIGHT_MARGIN_INCH = 0.45
TOP_MARGIN_INCH = 0.8
BOTTOM_MARGIN_INCH = 0.35
PAIR_GAP_INCH = 0.8


def as_grid(grid):
    arr = np.asarray(grid, dtype=int)
    if arr.ndim != 2:
        raise ValueError(f"Expected a 2D grid, got shape {arr.shape}")
    if arr.size == 0:
        raise ValueError("Grid is empty.")
    return arr


def _grid_extent_inch(grid):
    arr = as_grid(grid)
    rows, cols = arr.shape
    return cols * CELL_SIZE_INCH, rows * CELL_SIZE_INCH


def draw_arc_grid(ax, grid, title=""):
    # Copy so imshow does not keep a live view of upstream buffers (avoids duplicate renders in Jupyter).
    arr = np.asarray(as_grid(grid), dtype=int).copy()
    ax.imshow(arr, cmap=ARC_CMAP, norm=ARC_NORM, interpolation="nearest")
    ax.set_title(f"{title}\n{arr.shape[0]}x{arr.shape[1]}", fontsize=11)
    ax.set_xticks(np.arange(-0.5, arr.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, arr.shape[0], 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=0.8)
    ax.tick_params(which="both", bottom=False, left=False, labelbottom=False, labelleft=False)
    ax.set_aspect("equal")


def _add_grid_axes(fig, grid, x_inch, y_inch, fig_w, fig_h, max_rows, title=""):
    arr = as_grid(grid)
    rows, cols = arr.shape
    width_inch = cols * CELL_SIZE_INCH
    height_inch = rows * CELL_SIZE_INCH
    vertical_offset = (max_rows - rows) * CELL_SIZE_INCH / 2

    ax = fig.add_axes([
        x_inch / fig_w,
        (y_inch + vertical_offset) / fig_h,
        width_inch / fig_w,
        height_inch / fig_h,
    ])
    draw_arc_grid(ax, arr, title=title)
    return ax


def show_grid_pair(input_grid, output_grid=None, title=""):
    input_arr = as_grid(input_grid)
    input_w, input_h = _grid_extent_inch(input_arr)

    if output_grid is None:
        fig_w = LEFT_MARGIN_INCH + input_w + RIGHT_MARGIN_INCH
        fig_h = TOP_MARGIN_INCH + input_h + BOTTOM_MARGIN_INCH
        fig = plt.figure(figsize=(fig_w, fig_h))
        _add_grid_axes(
            fig,
            input_arr,
            LEFT_MARGIN_INCH,
            BOTTOM_MARGIN_INCH,
            fig_w,
            fig_h,
            max_rows=input_arr.shape[0],
            title=title or "Grid",
        )
    else:
        output_arr = as_grid(output_grid)
        output_w, _ = _grid_extent_inch(output_arr)
        max_rows = max(input_arr.shape[0], output_arr.shape[0])
        fig_w = LEFT_MARGIN_INCH + input_w + PAIR_GAP_INCH + output_w + RIGHT_MARGIN_INCH
        fig_h = TOP_MARGIN_INCH + max_rows * CELL_SIZE_INCH + BOTTOM_MARGIN_INCH
        fig = plt.figure(figsize=(fig_w, fig_h))
        _add_grid_axes(
            fig,
            input_arr,
            LEFT_MARGIN_INCH,
            BOTTOM_MARGIN_INCH,
            fig_w,
            fig_h,
            max_rows=max_rows,
            title=f"{title} Input".strip(),
        )
        _add_grid_axes(
            fig,
            output_arr,
            LEFT_MARGIN_INCH + input_w + PAIR_GAP_INCH,
            BOTTOM_MARGIN_INCH,
            fig_w,
            fig_h,
            max_rows=max_rows,
            title=f"{title} Output".strip(),
        )

    plt.show()


def show_arc_examples(examples, section_title):
    if not examples:
        print(f"{section_title}: no examples")
        return

    for idx, example in enumerate(examples, start=1):
        show_grid_pair(example["input"], example.get("output"), title=f"{section_title} {idx}")

In [3]:
def find_arc_task_path(task_id, data_root=ARC_DATA_ROOT):
    data_root = Path(data_root).expanduser().resolve()
    task_stem = Path(str(task_id)).stem.lower()
    matches = sorted(data_root.rglob(f"{task_stem}.json"))
    if not matches:
        raise FileNotFoundError(f"Could not find ARC task: {task_id}")
    return matches[0]


def load_arc_task(task_id, data_root=ARC_DATA_ROOT):
    task_path = find_arc_task_path(task_id, data_root=data_root)
    return load_arc_task_path(task_path, data_root=data_root)


def load_arc_task_path(task_path, data_root=ARC_DATA_ROOT):
    task_path = Path(task_path).expanduser().resolve()
    with task_path.open("r", encoding="utf-8") as fh:
        task = json.load(fh)
    return task_path, task


def visualize_arc_task(task_id, data_root=ARC_DATA_ROOT):
    task_path, task = load_arc_task(task_id, data_root=data_root)
    return visualize_arc_task_data(task_path, task, data_root=data_root)


def visualize_arc_task_path(task_path, data_root=ARC_DATA_ROOT):
    task_path, task = load_arc_task_path(task_path, data_root=data_root)
    return visualize_arc_task_data(task_path, task, data_root=data_root)


def visualize_arc_task_data(task_path, task, data_root=ARC_DATA_ROOT):
    data_root = Path(data_root).expanduser().resolve()
    task_path = Path(task_path).expanduser().resolve()
    try:
        rel_path = task_path.relative_to(data_root)
    except ValueError:
        rel_path = task_path
    display(Markdown(f"### ARC Task `{task_path.stem}`\n`{rel_path}`"))
    show_arc_examples(task.get("train", []), "Train")
    show_arc_examples(task.get("test", []), "Test")
    return {
        "task_id": task_path.stem,
        "task_path": task_path,
        "train_examples": len(task.get("train", [])),
        "test_examples": len(task.get("test", [])),
    }

In [ ]:
arc_task_id = "b15fca0b"  # 改成任意 ARC 题号即可
visualize_arc_task(arc_task_id)

## ARC1 随机 50 题可视化

从 `ARC1_DATA_ROOT` 递归收集 ARC1 的 `training` 和 `evaluation` 题目 JSON，使用固定随机种子抽取 50 题，并按现有 ARC 标准格式依次显示 train/test 示例。

In [ ]:
def list_arc_task_paths(data_root=ARC1_DATA_ROOT):
    data_root = Path(data_root).expanduser().resolve()
    if not data_root.exists():
        raise FileNotFoundError(data_root)
    return sorted(path for path in data_root.rglob("*.json") if path.is_file())


def sample_arc_task_paths(data_root=ARC1_DATA_ROOT, sample_size=50, seed=0):
    task_paths = list_arc_task_paths(data_root=data_root)
    if not task_paths:
        raise ValueError(f"No ARC task json files found under {data_root}")
    if sample_size > len(task_paths):
        raise ValueError(f"sample_size={sample_size} exceeds available tasks={len(task_paths)}")
    rng = random.Random(seed)
    return rng.sample(task_paths, sample_size)


def visualize_random_arc_tasks(data_root=ARC1_DATA_ROOT, sample_size=50, seed=0):
    data_root = Path(data_root).expanduser().resolve()
    sampled_paths = sample_arc_task_paths(data_root=data_root, sample_size=sample_size, seed=seed)
    display(Markdown(
        f"## ARC1 random sample visualization\n"
        f"- Data root: `{data_root}`\n"
        f"- Sample size: {sample_size}\n"
        f"- Random seed: {seed}"
    ))

    summaries = []
    for idx, task_path in enumerate(sampled_paths, start=1):
        display(Markdown(f"## ARC1 sample {idx}/{sample_size}: `{task_path.stem}`"))
        summaries.append(visualize_arc_task_path(task_path, data_root=data_root))
    return summaries


ARC1_RANDOM_SAMPLE_SIZE = 50
ARC1_RANDOM_SEED = 0

arc1_random_summaries = visualize_random_arc_tasks(
    data_root=ARC1_DATA_ROOT,
    sample_size=ARC1_RANDOM_SAMPLE_SIZE,
    seed=ARC1_RANDOM_SEED,
)
arc1_random_summaries

In [4]:
def load_module_from_path(module_path):
    module_path = Path(module_path).expanduser().resolve()
    module_key = hashlib.sha1(str(module_path).encode("utf-8")).hexdigest()[:12]
    module_name = f"arc_viz_{module_path.stem}_{module_key}"
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load module spec for {module_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def find_arc2_solver_path(task_id, arc2_root=ARC2_ROOT):
    task_stem = Path(str(task_id)).stem.lower()
    solver_path = arc2_root / f"{task_stem}.py"
    if not solver_path.exists():
        raise FileNotFoundError(f"Could not find arc2 solver for task id: {task_id}")
    return solver_path


def discover_code_files(root_path, pattern="*.py"):
    root = Path(root_path).expanduser().resolve()
    if not root.exists():
        raise FileNotFoundError(root)
    if root.is_file():
        return [root]
    return sorted(path for path in root.rglob(pattern) if path.name != "__init__.py")


def supports_visualization(module):
    return callable(getattr(module, "generate_input", None)) and callable(getattr(module, "main", None))


def call_generate_input(generate_input_fn, seed=0):
    signature = inspect.signature(generate_input_fn)
    required = [
        param
        for param in signature.parameters.values()
        if param.kind in (inspect.Parameter.POSITIONAL_ONLY, inspect.Parameter.POSITIONAL_OR_KEYWORD)
        and param.default is inspect.Signature.empty
    ]
    if not required:
        return generate_input_fn()
    if len(required) == 1:
        param = required[0]
        if param.kind is inspect.Parameter.POSITIONAL_ONLY:
            return generate_input_fn(seed)
        return generate_input_fn(**{param.name: seed})
    raise TypeError(f"Unsupported generate_input signature: {signature}")


def generate_io_pair(module, seed=0):
    input_grid = as_grid(call_generate_input(module.generate_input, seed=seed))
    output_grid = as_grid(module.main(input_grid.copy()))
    return input_grid, output_grid


In [5]:
def visualize_module_file(module_path, seed=0):
    module_path = Path(module_path).expanduser().resolve()
    module = load_module_from_path(module_path)
    if not supports_visualization(module):
        print(f"Skip {module_path}: missing generate_input() or main().")
        return False

    input_grid, output_grid = generate_io_pair(module, seed=seed)
    display(Markdown(f"### `{module_path.name}`\n`{module_path}`"))
    show_grid_pair(input_grid, output_grid, title=module_path.stem)
    return True


def visualize_arc2_task(task_id, seed=0, arc2_root=ARC2_ROOT):
    solver_path = find_arc2_solver_path(task_id, arc2_root=arc2_root)
    module = load_module_from_path(solver_path)
    if not supports_visualization(module):
        raise ValueError(f"{task_id} has no generate_input() or main() for visualization")

    input_grid, output_grid = generate_io_pair(module, seed=seed)
    display(Markdown(f"### ARC2 Solver `{solver_path.stem}`\n`{solver_path}`"))
    show_grid_pair(input_grid, output_grid, title=solver_path.stem)
    return {
        "task_id": solver_path.stem,
        "solver_path": solver_path,
        "input_shape": input_grid.shape,
        "output_shape": output_grid.shape,
    }


def parse_task_ids_from_text(text: str) -> list[str]:
    """从粘贴文本中提取 8 位十六进制题号；保留首次出现顺序并去重。"""
    raw = re.findall(r"\b[0-9a-fA-F]{8}\b", text)
    seen: set[str] = set()
    ordered: list[str] = []
    for x in raw:
        k = x.lower()
        if k not in seen:
            seen.add(k)
            ordered.append(k)
    return ordered


def visualize_code_files(code_files, *, root=None, seed=0, num_examples=1):
    paths = [Path(p).expanduser().resolve() for p in code_files]
    root = Path(root).expanduser().resolve() if root is not None else None

    def rel_path_for(cf: Path) -> Path:
        if root is not None and root.is_dir():
            try:
                return cf.relative_to(root)
            except ValueError:
                pass
        return Path(cf.name)

    visualized = 0
    pairs_visualized = 0
    skipped = []
    errors = []

    for code_file in paths:
        try:
            module = load_module_from_path(code_file)
            if not supports_visualization(module):
                skipped.append((code_file, "missing generate_input() or main()"))
                continue

            rel_path = rel_path_for(code_file)
            display(Markdown(f"### `{rel_path}`"))
            for example_idx in range(num_examples):
                example_seed = seed + example_idx
                input_grid, output_grid = generate_io_pair(module, seed=example_seed)
                show_grid_pair(input_grid, output_grid, title=f"{code_file.stem} #{example_idx + 1}")
                pairs_visualized += 1
            visualized += 1
        except Exception as exc:
            errors.append((code_file, exc))

    where = f" in {root}" if root is not None else ""
    print(f"Scanned {len(paths)} files{where}")
    print(f"Visualized files: {visualized}")
    print(f"Visualized pairs: {pairs_visualized}")

    if skipped:
        print(f"Skipped: {len(skipped)}")
        for code_file, reason in skipped:
            print(f"  - {rel_path_for(code_file)}: {reason}")

    if errors:
        print(f"Errors: {len(errors)}")
        for code_file, exc in errors:
            print(f"  - {rel_path_for(code_file)}: {type(exc).__name__}: {exc}")

    return {
        "root": root,
        "scanned": len(paths),
        "visualized": visualized,
        "examples_per_file": num_examples,
        "pairs_visualized": pairs_visualized,
        "skipped": skipped,
        "errors": errors,
    }


def visualize_code_path(root_path, seed=0, max_files=None, pattern="*.py", num_examples=1):
    root = Path(root_path).expanduser().resolve()
    code_files = discover_code_files(root, pattern=pattern)
    if max_files is not None:
        code_files = code_files[:max_files]
    return visualize_code_files(code_files, root=root, seed=seed, num_examples=num_examples)


In [ ]:
move_to_match_ids_path = Path("/home/lixinyu/snap/new/annotations/move_to_match.txt")
move_to_match_data_root = Path("/home/lixinyu/snap/new/ARC-data/ARC-AGI-2/data")

# None 表示可视化 txt 中全部题号；调试时可改成整数，例如 5。
move_to_match_max_tasks = None

move_to_match_task_ids = [
    line.strip().lower()
    for line in move_to_match_ids_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
if move_to_match_max_tasks is not None:
    move_to_match_task_ids = move_to_match_task_ids[:move_to_match_max_tasks]

display(Markdown(
    f"## move_to_match 题目可视化\n"
    f"- 题号文件：`{move_to_match_ids_path}`\n"
    f"- 数据路径：`{move_to_match_data_root}`\n"
    f"- 本次可视化：{len(move_to_match_task_ids)} 题"
))

move_to_match_missing = []
for task_idx, task_id in enumerate(move_to_match_task_ids, start=1):
    try:
        display(Markdown(f"## move_to_match {task_idx}/{len(move_to_match_task_ids)}: `{task_id}`"))
        visualize_arc_task(task_id, data_root=move_to_match_data_root)
    except FileNotFoundError:
        move_to_match_missing.append(task_id)
        display(Markdown(f"**Missing ARC task data:** `{task_id}`"))

if move_to_match_missing:
    print(f"Missing {len(move_to_match_missing)} tasks:", move_to_match_missing)
else:
    print(f"Visualized all {len(move_to_match_task_ids)} move_to_match tasks.")

In [ ]:
x=['291dc1e1','271d71e2','446ef5d2','247ef758','221dfab4']
for i in x:
    arc2_task_id = i  # 改成 arc2 的题号
    for i in range(0,5):
        visualize_arc2_task(arc2_task_id, seed=i)

In [ ]:
arc2_task_id = "269e22fb"  # 改成 arc2 的题号
for i in range(0,4):
    visualize_arc2_task(arc2_task_id, seed=i)

In [ ]:
variants_path = "/home/lixinyu/snap/new/ceshi/code/6ffbe589"  
# variants_path = "/home/lixinyu/snap/new/ceshi/code/78332cb0/D14"  
variants_seed = 0
variants_examples = 4
max_files = None  # 例如设为 5；None 表示该路径下全部匹配的 code.py

variants_summary = visualize_code_path(
    variants_path,
    seed=variants_seed,
    max_files=max_files,
    pattern="code.py",
    num_examples=variants_examples,
)
variants_summary

输出ARC2题目代码的可视化

In [8]:
arc2_solver_select_ids_text = """
aa4ec2a5 abc82100 b0039139 b10624e5 b5ca7ac4 b99e7126 b9e38dc0 c4d067a0 c7f57c3e cb2d8a2c cbebaa4b d35bdbdc d59b0160 d8e07eb2 da515329 db0c5428 db695cfb dbff022c dd6b8c4b de809cff dfadab01 e12f9a14 e3721c99 e376de54 e8686506 e87109e9 edb79dae eee78d87 f560132c f931b4a8 faa9f03d 28a6681f
"""

In [ ]:
arc2_solver_select_path = BASE_DIR / "new/arc2_solver_select"
# 粘贴题号：支持逗号/换行/空格/混排；自动识别 8 位十六进制题号
arc2_solver_select_seed = 0
arc2_solver_select_examples = 4

_task_ids = parse_task_ids_from_text(arc2_solver_select_ids_text)
if not _task_ids:
    print("未从 arc2_solver_select_ids_text 中识别到 8 位十六进制题号（可逗号/换行/空格混排）。")

arc2_solver_select_py_paths = []
arc2_solver_select_missing = []
for _tid in _task_ids:
    _p = arc2_solver_select_path / f"{_tid}.py"
    if _p.is_file():
        arc2_solver_select_py_paths.append(_p)
    else:
        arc2_solver_select_missing.append(_tid)

if arc2_solver_select_missing:
    print(
        f"目录下缺少对应 .py（共 {len(arc2_solver_select_missing)} 个）: "
        f"{arc2_solver_select_missing}"
    )

arc2_solver_select_summary = visualize_code_files(
    arc2_solver_select_py_paths,
    root=arc2_solver_select_path,
    seed=arc2_solver_select_seed,
    num_examples=arc2_solver_select_examples,
)
arc2_solver_select_summary